# Workshop 8. Yes/no variables and the linear probability model (solutions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture08/workshop08_solutions.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the lecture's Part 1 in code, fully worked: run it, answer the questions and change one thing. Tasks 2 to
4 start from an empty cell: age groups as a category with a reference, an interaction between two yes/no variables,
and a linear probability model for churn. The extension tasks are for those who have time left. Exercises in the
style of the textbook, with small numbers and no data, are in the TA's separate notebook for this session (link on
Moodle).

Each core task has two hints: the first names the statistical step, the second names the functions. Try the task
before you open them. The toolbox says where you have seen each function before, so you can look it up in that
notebook. The code patterns, the slides and the lecture notebook are your main references. You are welcome to use an
LLM, but every task can be solved without one, and you should be able to explain the statistics in your answers
yourself.

In every task, say what y and the x's are, and in which units, before you fit anything. Say which category is the
reference, and whether a number is in percent or in percentage points. Each answer is in words a manager can follow,
with the numbers it rests on.

## Toolbox

The functions you need today, and where you have seen them in use.

| Function | What it does | Seen in |
|---|---|---|
| `smf.ols("y ~ x1 + x2", data=df).fit(cov_type="HC1")` | Fits a regression with robust standard errors | Lecture 7, Parts 2 and 3 |
| `model.params`, `model.bse`, `model.pvalues`, `model.conf_int()` | Coefficients, standard errors, p-values, 95% intervals | Lecture 6; Lecture 8, Part 1 |
| `df.groupby("g")["y"].mean()` | The mean of y in each group | Lecture 4; Lecture 8, Part 1 |
| `C(region)` in a formula | A category turned into dummies; the first value is the reference | Lecture 8, Part 2 |
| `C(region, Treatment('Nordjylland'))` | The same with a chosen reference category | Lecture 8, Part 2 |
| `a * b`, `a:b` in a formula | Both main terms and their product; the product only | Lecture 8, Part 3 |
| `np.exp(b)` | e to the power b; `100 * (np.exp(b) - 1)` is the exact percent for a coefficient in a model for ln y | Lecture 7, Part 2; Lecture 8, Part 1 |
| `pd.cut(df["x"], bins=[...], labels=[...])` | Cuts a number into ordered groups | New today: see the first code pattern |
| `(df["x"] < 2).astype(int)` | A 0/1 column from a condition | Lecture 8, Part 2 (`east`) |
| `model.fittedvalues` | ŷ, one per row; in an LPM the predicted probabilities | Lecture 7, Part 1; Lecture 8, Part 4 |
| `model.predict(pd.DataFrame({...}))` | Predictions; one column for every x in the formula | Lecture 7, Part 4; Lecture 8, Part 4 |

## Code patterns

Two complete patterns for today. Replace `df`, `y`, `x`, `g`, `d1`, `d2` and the labels with your data and column
names. The patterns are templates; they do not run as they stand.

**Groups from a number, a category with a reference, and an interaction of two dummies**

```python
# groups from a number: bins are the edges; a value goes into the group above its lower edge and up to and
# including its upper edge, so 0 here would fall outside every group
df["x_group"] = pd.cut(df["x"], bins=[0, 10, 20, 30], labels=["1-10", "11-20", "21-30"])
print(df["x_group"].value_counts().sort_index())

model = smf.ols("y ~ C(x_group) + x2", data=df).fit(cov_type="HC1")        # reference: the first group
model2 = smf.ols("y ~ C(x_group, Treatment('11-20')) + x2", data=df).fit(cov_type="HC1")   # another reference

# an interaction of two 0/1 columns: d1 * d2 gives d1, d2 and d1:d2
df["d2"] = (df["z"] < 5).astype(int)
inter = smf.ols("log_y ~ d1 * d2 + x2", data=df).fit(cov_type="HC1")
b = inter.params
gap_0 = b["d1"]                    # gap between d1 = 1 and d1 = 0 when d2 = 0
gap_1 = b["d1"] + b["d1:d2"]       # the same gap when d2 = 1
print(f"{100 * (np.exp(gap_0) - 1):.1f}% and {100 * (np.exp(gap_1) - 1):.1f}%")
```

**A linear probability model with robust standard errors, predictions, and values outside 0 to 1**

```python
lpm = smf.ols("y01 ~ d1 + C(g) + x", data=df).fit(cov_type="HC1")         # y01 is 0 or 1
print((100 * lpm.params).round(1))                                         # percentage points

classical = smf.ols("y01 ~ d1 + C(g) + x", data=df).fit()
print(pd.DataFrame({"se classical": classical.bse, "se robust": lpm.bse}).round(4))

p_hat = lpm.fittedvalues
print(f"Below 0: {(p_hat < 0).sum()}, above 1: {(p_hat > 1).sum()}")

new = pd.DataFrame({"d1": [0, 1], "g": ["A", "B"], "x": [3, 8]})           # one column for every x
print(lpm.predict(new).round(2))
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 RetailCorp customers drawn at random
customers["log_spend"] = np.log(customers["spend_2025"])
customers.head()

## Core tasks

### Task 1. Do club members spend more? (worked)

The lecture's Part 1 in code. Run the cells and answer the questions below them.

y = `spend_2025` (DKK in 2025), then `log_spend` (its ln); the variable of interest is `member` (1 if in the loyalty
club, 0 otherwise); the controls are `tenure_years`, `age` and `visits_2025`.

In [ ]:
means = customers.groupby("member")["spend_2025"].mean()
print(means.round(2))

m_dkk = smf.ols("spend_2025 ~ member", data=customers).fit()
m_dkk_r = smf.ols("spend_2025 ~ member", data=customers).fit(cov_type="HC1")
print(m_dkk.params.round(2))
print(f"t = {m_dkk.tvalues['member']:.2f} (classical, as in the lecture), z = {m_dkk_r.tvalues['member']:.2f} (HC1)")

In [ ]:
controls = "tenure_years + age + visits_2025"      # step 3: change to "tenure_years + age"

m_log = smf.ols(f"log_spend ~ member + {controls}", data=customers).fit(cov_type="HC1")
b = m_log.params["member"]
low, high = m_log.conf_int().loc["member"]
print(m_log.params.round(4))
print(f"member: about {100 * b:.1f}%, exactly {100 * (np.exp(b) - 1):.1f}% "
      f"(95% interval {100 * (np.exp(low) - 1):.1f}% to {100 * (np.exp(high) - 1):.1f}%)")

**Questions**

1. What do the intercept and the member coefficient of `m_dkk` mean? Compare them with the two group means.
2. Read the member coefficient of `m_log` as a percent, about and exactly, with its interval. Why do the about and
   the exact values differ here, when they were almost equal for tenure in Lecture 7?
3. Which customers does the member coefficient in `m_log` compare?

**Change one thing.** In the second cell, change `controls` to `"tenure_years + age"` (visits left out) and run it.
How does the member coefficient change, and why?

**Answer.**
1. The intercept, 1,849.21 DKK, is the mean spend of non-members, the group coded 0. The member coefficient, 773.27
   DKK, is the difference between the two means (2,622.48 − 1,849.21). A regression on one dummy reproduces the two
   group means. t = 8.30 with classical standard errors, as in the lecture, the same as the pooled t test
   from Lecture 4; z = 8.10 with robust standard errors.
2. b = 0.2197: about 22.0%, exactly 24.6% higher spend (95% interval 16.4% to 33.4%). The exact percent is
   100 × (exp(b) − 1), since a difference of b in ln spend multiplies spend by e to the power b. The about version
   100 × b is close only for small b, below about 0.1; tenure's 0.04 was small, 0.22 is not.
3. Members and non-members with the same tenure, age and visits. It describes customers as they are: customers
   chose to join the club.

**Without visits.** The member coefficient rises to 0.3201: exactly 37.7% (28.4% to 47.7%). Members visit more (34.4
against 26.3 visits), and visits go with higher spend, so without visits in the model part of the visits difference
shows up as a membership difference: omitted variable reasoning, as in Lecture 7.

### Task 2. Age groups as a category

In the lecture, age entered the model as a number, which assumes that every year of age goes with the same
difference in ln spend. Here you let the data show the steps instead.

1. Make a column `age_group` with `pd.cut`: 18-34, 35-49, 50-64 and 65+ (bins at 17, 34, 49, 64 and 85). How many
   customers are in each group?
2. Fit ln spend on `C(age_group)`, `member`, `tenure_years` and `visits_2025`, with robust standard errors. Which
   group is the reference? Read the three age group coefficients as exact percents.
3. Are the steps from one group to the next about even? Compare with a model with `age` as a number: how large is the
   age coefficient for 15 years of age, about the distance between the group averages? Which version would you
   report?
4. Make 35-49 the reference. What changes, and what does not?

<details>
<summary>Hint 1</summary>

pd.cut puts a customer in the group above its lower edge and up to and including its upper edge, so bins starting at
17 include the 18-year-olds. For a column made with pd.cut, the reference is the first group in the order of the bins,
and each coefficient is a difference from it. For question 3, look at
the differences between neighboring group coefficients, and at the average age in each group.
</details>

<details>
<summary>Hint 2</summary>

The first code pattern. `customers.groupby("age_group", observed=True)["age"].mean()` gives the average age in each
group. For question 4: `C(age_group, Treatment('35-49'))`.
</details>

In [ ]:
customers["age_group"] = pd.cut(customers["age"], bins=[17, 34, 49, 64, 85],
                                labels=["18-34", "35-49", "50-64", "65+"])
print(customers["age_group"].value_counts().sort_index())
print(customers.groupby("age_group", observed=True)["age"].mean().round(1))

In [ ]:
m_groups = smf.ols("log_spend ~ C(age_group) + member + tenure_years + visits_2025",
                   data=customers).fit(cov_type="HC1")
for name in m_groups.params.index:
    if name.startswith("C(age_group)"):
        b = m_groups.params[name]
        print(f"{name:25s} {b:.4f}   exactly {100 * (np.exp(b) - 1):.1f}%   p = {m_groups.pvalues[name]:.3f}")

In [ ]:
m_number = smf.ols("log_spend ~ age + member + tenure_years + visits_2025", data=customers).fit(cov_type="HC1")
b_age = m_number.params["age"]
print(f"age as a number: {b_age:.4f} per year, {15 * b_age:.3f} per 15 years")

In [ ]:
m_ref = smf.ols("log_spend ~ C(age_group, Treatment('35-49')) + member + tenure_years + visits_2025",
                data=customers).fit(cov_type="HC1")
print(m_ref.params.round(4))
print(f"R2: {m_groups.rsquared:.4f} and {m_ref.rsquared:.4f}")

1. 181 customers are 18-34, 428 are 35-49, 297 are 50-64 and 94 are 65+. The average ages in the groups are 27.6,
   42.8, 55.8 and 71.8.
2. The reference is 18-34, the first group. Compared with customers aged 18-34 with the same membership, tenure and
   visits, customers aged 35-49 spend 33.1% more (0.2857), 50-64 56.1% more (0.4456) and 65+ 109.5% more (0.7394),
   all p < 0.001.
3. The steps between neighboring groups are 0.286, 0.160 and 0.294 on the ln scale, for about 13 to 16 years of age
   each. With age as a number the coefficient is 0.0174 per year, 0.260 for 15 years. The steps wobble around that,
   with no clear pattern of flattening or speeding up, so age as a number is a fair summary here: one coefficient
   instead of three, under the assumption of equal steps that the groups do not contradict. Reporting the groups is
   also defensible if a manager thinks in age groups; say which you chose and why.
4. With 35-49 as the reference, 18-34 is −0.2857, 50-64 +0.1600 and 65+ +0.4537. Every coefficient is now a
   difference from 35-49, but the difference between any two groups is the same (0.4456 − 0.2857 = 0.1599, which is
   0.1600 up to rounding). Member
   (0.2199), tenure, visits, the predictions and R² (0.3041) do not change.

### Task 3. Is the club gap the same for new customers?

In the lecture, the membership gap narrowed with each year of tenure. Here you ask the same question with a yes/no
variable: is the gap different for new customers?

1. Make a column `new_cust` that is 1 if the customer has been a customer for less than 2 years, and 0 otherwise. What
   share of the customers is new?
2. Fit ln spend on `member * new_cust`, `age` and `visits_2025`, with robust standard errors. Write the 2×2 table of the
   gaps: the membership gap for established customers and for new customers, in exact percent.
3. Is the difference between the two gaps clear, or could it be chance?
4. Compare with the lecture's member × tenure model. What does the yes/no version make easier to report, and what
   does it lose?

<details>
<summary>Hint 1</summary>

With member * new_cust, the member coefficient is the gap when new_cust = 0, and the gap when new_cust = 1 adds the interaction
coefficient. The interaction coefficient is the difference between the two gaps, so its test answers question 3.
</details>

<details>
<summary>Hint 2</summary>

`(customers["tenure_years"] < 2).astype(int)`, `.mean()`, and the interaction part of the first code pattern:
`b["member"]` and `b["member"] + b["member:new_cust"]`, each through `100 * (np.exp(...) - 1)`.
</details>

In [ ]:
customers["new_cust"] = (customers["tenure_years"] < 2).astype(int)
print(f"Share new: {customers['new_cust'].mean():.3f}")
print(pd.crosstab(customers["member"], customers["new_cust"]))

In [ ]:
m_new = smf.ols("log_spend ~ member * new_cust + age + visits_2025", data=customers).fit(cov_type="HC1")
b = m_new.params
print(b.round(4))
print(f"p for member:new_cust = {m_new.pvalues['member:new_cust']:.4f}")
print(m_new.conf_int().loc["member:new_cust"].round(3))
print(f"Gap, established customers: {100 * (np.exp(b['member']) - 1):.1f}%")
print(f"Gap, new customers:         {100 * (np.exp(b['member'] + b['member:new_cust']) - 1):.1f}%")

1. 41.4% of the customers have been customers for less than 2 years (414 of 1,000; 139 of them are members).
2. At the same age and visits:

   | | Established (new_cust = 0) | New (new_cust = 1) |
   |---|---|---|
   | Membership gap, ln scale | 0.1541 | 0.1541 + 0.2054 = 0.3595 |
   | Membership gap, exactly | 16.7% | 43.3% |

3. The interaction coefficient, 0.2054, is the difference between the two gaps on the ln scale; p = 0.003 (95%
   interval 0.069 to 0.342). The gap is clearly larger for new customers.
4. It is the same pattern as in the lecture: membership goes with a larger spend gap for newer customers. The yes/no
   version is easy to report (two gaps, one for each group) and reads as a 2×2 table. It loses the gradual
   narrowing: everyone under 2 years counts the same, everyone above as well, and the result depends on where the
   line is drawn (Extension 2).

### Task 4. Which customers stop buying?

y = `churned` (1 if no order in January to June 2026). Use the age groups from Task 2 in place of age.

1. Fit a linear probability model of `churned` on `member`, `C(channel)`, `tenure_years`, `visits_2025` and
   `C(age_group)`, with robust standard errors. Read the member and Web coefficients in percentage points.
2. Do the age groups differ in churn, at the same membership, channel, tenure and visits?
3. Compare the classical and the robust standard error of member. Why do we use the robust one?
4. How many predicted probabilities lie outside 0 to 1? Why can that happen?
5. Predict the probability of churn for two customers: (A) a web customer who is not a member, aged 25, with 1 year
   and 15 visits; (B) an app user who is a member, aged 55, with 3 years and 30 visits. Write the difference between
   them in percentage points.

<details>
<summary>Hint 1</summary>

LPM coefficients times 100 are percentage points. Unequal scatter is built into a model with a 0/1 outcome, so the
classical standard errors rest on an assumption that cannot hold. A straight line can go below 0 or above 1.
</details>

<details>
<summary>Hint 2</summary>

The second code pattern. In the prediction table, `age_group` takes the labels from Task 2, for example `"18-34"`,
and `channel` takes `"Web"` or `"App"`.
</details>

In [ ]:
lpm = smf.ols("churned ~ member + C(channel) + tenure_years + visits_2025 + C(age_group)",
              data=customers).fit(cov_type="HC1")
print((100 * lpm.params).round(1))
print(lpm.pvalues.round(3))

In [ ]:
classical = smf.ols("churned ~ member + C(channel) + tenure_years + visits_2025 + C(age_group)",
                    data=customers).fit()
print(f"member: classical se {classical.bse['member']:.4f}, robust se {lpm.bse['member']:.4f}")

In [ ]:
p_hat = lpm.fittedvalues
print(f"Below 0: {(p_hat < 0).sum()}, above 1: {(p_hat > 1).sum()}, lowest {p_hat.min():.2f}, highest {p_hat.max():.2f}")

In [ ]:
new = pd.DataFrame({"member": [0, 1], "channel": ["Web", "App"], "tenure_years": [1, 3],
                    "visits_2025": [15, 30], "age_group": ["18-34", "50-64"]})
print(lpm.predict(new).round(3))

1. At the same channel, tenure, visits and age group, members are 10.7 percentage points less likely to churn
   (−0.1070, p < 0.001), and web customers 10.2 points more likely than app users (0.1019, p = 0.001). These are
   almost the same numbers as in the lecture, where age entered as a number.
2. No clear differences: compared with 18-34, the age groups differ by +4.1, +0.7 and −4.6 points, with p = 0.30,
   0.88 and 0.45. Age adds little to churn once membership, channel, tenure and visits are in the model, also in the
   lecture (p = 0.22).
3. Classical 0.0294, robust 0.0284. They are close here, but the outcome is 0 or 1, so its variance p(1 − p) differs
   between customers whose predicted probabilities differ. The classical standard errors assume equal scatter, which
   cannot hold, so we use the robust ones in every LPM.
4. 39 predicted probabilities are below 0 (the lowest is −0.19) and none above 1 (the highest is 0.63). The line is
   straight, so it keeps falling for customers with long tenure and many visits, where a probability cannot go below
   0.
5. Customer A: 0.527, a 53% probability of churning. Customer B: 0.182, 18%. B's probability is about 35 percentage
   points lower: 10.7 from membership, 10.2 from the app, 5.6 from two more years and 8.7 from 15 more visits, and
   0.7 higher from the age group (rounded).

## Extension tasks

### Extension 1. Does region matter at all?

In the lecture, no single region differed clearly from Hovedstaden. Whether region matters at all is a joint
question about the four region coefficients together. Fit ln spend on `member`, `C(region)`, `tenure_years`, `age`
and `visits_2025` with robust standard errors, and test that all four region coefficients are 0 with
`model.wald_test("C(region)[T.Midtjylland] = 0, C(region)[T.Nordjylland] = 0, ...", scalar=True)` (all four). What
do you conclude?

In [ ]:
m_reg = smf.ols("log_spend ~ member + C(region) + tenure_years + age + visits_2025", data=customers).fit(cov_type="HC1")
names = [n for n in m_reg.params.index if n.startswith("C(region)")]
hypothesis = ", ".join(f"{n} = 0" for n in names)
print(hypothesis)
test = m_reg.wald_test(hypothesis, scalar=True)
print(f"Wald statistic = {test.statistic:.2f} with {len(names)} degrees of freedom, p = {test.pvalue:.2f}")

The Wald statistic is 3.75 with 4 degrees of freedom, p = 0.44 (statsmodels reports it as a χ² test here).
The data are consistent with the same spend in all five regions, at the same membership, tenure, age and visits. This
supports leaving region out, which the lecture did because region barely changed the member coefficient. One test of
all four coefficients answers the question; four separate p-values do not, because each depends on the reference.

### Extension 2. Where to draw the line

Repeat Task 3 with new customers defined as less than 5 years instead of less than 2. How do the two gaps and the
interaction change? What does that say about turning a number into a yes/no variable?

In [ ]:
customers["new5"] = (customers["tenure_years"] < 5).astype(int)
print(f"Share new: {customers['new5'].mean():.3f}")
m_new5 = smf.ols("log_spend ~ member * new5 + age + visits_2025", data=customers).fit(cov_type="HC1")
b = m_new5.params
print(b.round(4))
print(f"p for member:new5 = {m_new5.pvalues['member:new5']:.4f}")
print(f"Gap, 5 years or more: {100 * (np.exp(b['member']) - 1):.1f}%")
print(f"Gap, under 5 years:   {100 * (np.exp(b['member'] + b['member:new5']) - 1):.1f}%")

With a line at 5 years, 75.5% of the customers count as new. The gap is 8.3% for customers with 5 years or more and
32.5% for those under 5; the interaction is 0.2015 (p = 0.013). The conclusion stands (the gap is larger for newer
customers), but both gaps and the share in each group depend on the line. A yes/no version of a number is easier to
report, and the choice of line should be stated and, where it matters, checked. The lecture's member × tenure model
avoids the choice.